#### -----------------------------------------------------------------------------<br>Copyright (c) 2024, Lucid Vision Labs, Inc.
##### THE  SOFTWARE  IS  PROVIDED "AS IS",  WITHOUT  WARRANTY  OF  ANY  KIND,<br>EXPRESS  OR  IMPLIED,  INCLUDING  BUT  NOT  LIMITED  TO  THE  WARRANTIES<br>OF  MERCHANTABILITY,  FITNESS  FOR  A  PARTICULAR  PURPOSE  AND<br>NONINFRINGEMENT.  IN  NO  EVENT  SHALL  THE  AUTHORS  OR  COPYRIGHT  HOLDERS<br>BE  LIABLE  FOR  ANY  CLAIM,  DAMAGES  OR  OTHER  LIABILITY,  WHETHER  IN  AN<br>ACTION  OF  CONTRACT,  TORT  OR  OTHERWISE,  ARISING  FROM,  OUT  OF  OR  IN<br>CONNECTION  WITH  THE  SOFTWARE  OR  THE  USE  OR  OTHER  DEALINGS  IN  THE  SOFTWARE.<br>-----------------------------------------------------------------------------

In [1]:
import time

from arena_api.__future__.save import Writer
from arena_api.enums import PixelFormat
from arena_api.system import system
from arena_api.buffer import BufferFactory
from arena_api.__future__.save import _xWriter

#### Acquisition: Compressed Image Handling
>	This example demonstrates how to acquire and process compressed image data
	from the camera using the Arena SDK. The example includes
	steps to configure the camera, acquire a compressed image, process the
	image to decompress it, and save both the raw input and decompressed images.

In [2]:
tries = 0
tries_max = 6
sleep_time_secs = 10
while tries < tries_max:  # Wait for device for 60 seconds
	devices = system.create_device()
	if not devices:
		print(
			f'Try {tries+1} of {tries_max}: waiting for {sleep_time_secs} '
			f'secs for a device to be connected!')
		for sec_count in range(sleep_time_secs):
			time.sleep(1)
			print(f'{sec_count + 1 } seconds passed ',
				'.' * sec_count, end='\r')
		tries += 1
	else:
		print(f'Created {len(devices)} device(s)')
		device = system.select_device(devices)
		break
else:
	raise Exception(f'No device found! Please connect a device and run '
					f'the example again.')

print(f'Device used in the example:\n\t{device}')

Created 1 device(s)
  Only one device detected:  ('1c:0f:af:3f:55:a4', 'PHX064S-M', '', '169.254.165.85')
    Automatically selecting this device.
Device used in the example:
	('1c:0f:af:3f:55:a4', 'PHX064S-M', '', '169.254.165.85')


##### Enable stream auto negotiate packet size
>   Setting the stream packet size is done before starting the stream.
	Setting the stream to automatically negotiate packet size instructs the
	camera to receive the largest packet size that the system will allow.
	This generally increases frame rate and results in fewer interrupts per
	image, thereby reducing CPU load on the host system. Ethernet settings
	may also be manually changed to allow for a larger packet size.

In [3]:
tl_stream_nodemap = device.tl_stream_nodemap

tl_stream_nodemap['StreamAutoNegotiatePacketSize'].value = True

##### Enable stream packet resend
>   Enable stream packet resend before starting the stream. Images are sent
	from the camera to the host in packets using UDP protocol, which
	includes a header image number, packet number, and timestamp
	information. If a packet is missed while receiving an image, a packet
	resend is requested and this information is used to retrieve and
	redeliver the missing packet in the correct order.

In [4]:
tl_stream_nodemap['StreamPacketResendEnable'].value = True

#### Set features before streaming
>   Set PixelFormat to QOI_Mono8

In [5]:
# Get device nodemap
nodemap = device.nodemap

# Iterate through the PixelFormat enum and check for "QOI_Mono8"
node = nodemap.get_node('PixelFormat')
entries = node.enumentry_names
found = False
for e in entries:
	if (e == 'QOI_Mono8'):
		found = True
		break

if (not found):
	print(f'QOI_Mono8 is not available in the PixelFormat enumeration for this camera.\n')

# Get initial node values in order to return their values at the end of the example
pixel_format_initial = node.value
print(f"Initial PixelFormat value: {pixel_format_initial}")

Initial PixelFormat value: QOI_Mono8


In [6]:
# - PixelFormat to QOI_Mono8
pixel_format_setting = 'QOI_Mono8'
print(f'Setting pixel format to { pixel_format_setting }\n')
node.value = pixel_format_setting

Setting pixel format to QOI_Mono8



#### Start stream and grab images
>   - Starting stream with one buffer, grabbing one image.
>   - Printing the uncompressed size and compressed size for comparison.
>   - Saving compresed image (in PixelFormat QOI_Mono8) in its raw state and decompressing image to Mono8 and saving it.
>   - Must requeue buffer in order to prevent memory leaks.

In [7]:
device.start_stream(1)

print(f'Stream started with 1 buffer\n')

# Get compressed image
print(f'Get one image')
buffer = device.get_buffer()

Stream started with 1 buffer

Get one image


In [8]:
# Get compressed image size
compressed_image_size = buffer.size_filled
print(f'QOI_Mono8 compressed image size: {compressed_image_size} bytes\n')

QOI_Mono8 compressed image size: 3871460 bytes



>	Takes compresed image and saves the raw data

In [9]:
print(f'Saving compressed image')

# Set raw file save location
filename = r'images\py_acquisition_compressed_image_handling\CompressedImage.raw'

# Save function for .raw file
_xWriter.SaveRawData(filename, buffer.compressed_image_pdata, buffer.size_filled)
print(f'Image saved {filename}')

Saving compressed image
Image saved images\py_acquisition_compressed_image_handling\CompressedImage.raw


>	Decompresses image and saves it as a file

In [10]:
print(f'Decompressing to Mono8')

# Decompress image
png_buffer = BufferFactory.decompress_image(buffer)
print(f'Decompressed image')

# Get and print size for comparison
compressed_image_size = png_buffer.size_filled
print(f'Mono8 image size: {compressed_image_size} bytes\n')

# Save the decompressed image
writer = Writer.from_buffer(png_buffer)
writer.pattern = r'images/py_acquisition_compressed_image_handling/DecompressedImage.png'
writer.save(png_buffer)
print(f'Image saved {writer.saved_images[-1]}')

Decompressing to Mono8
Decompressed image
Mono8 image size: 6291456 bytes

Image saved c:\Users\mackenzie.dy\Documents\repo\software\arena_api\examples\images\py_acquisition_compressed_image_handling\DecompressedImage.png


>	Stops stream and prevents memory leaks

In [11]:
# Destroy converted buffer and requeues buffer to avoid memory leaks
BufferFactory.destroy(png_buffer)
device.requeue_buffer(buffer)

# Stop stream
print(f'Stopping stream')
device.stop_stream()

Stopping stream


>	Resets node values to initial values

In [12]:
node.value = pixel_format_initial

##### Clean up
> Destroy device. This call is optional and will automatically be
  called for any remaining devices when the system module is unloading.

In [13]:
system.destroy_device()
print(f'Destroyed all created devices')

Destroyed all created devices
